In [1]:
import numpy as np
import matplotlib.pyplot as plt

from itertools import chain, combinations

In [2]:

from sian.data import final_save_header
from sian.data import final_save_dataset
from sian.data import final_save_labels

from sian.utils import gettimestamp
from sian import powerset #MOVE BACK TO UTILS?
from itertools import product



base_save_dataset_path = "data/"
base_save_dataset_path = "../../../data/"
datetimestr = gettimestamp()

In [3]:

# N = 1000*1000
TRN_N = 1000*1000
TST_N = 10*1000
N = TRN_N + TST_N

# Ds = [10,30,50]
Ds = [10,20,30]
seeds=list(range(5))
Ds = [10]
seeds=list(range(1))

MAX_TUPLES_PER_BAND = 300
MAX_TUPLES_PER_BAND = 4  #04/16/2025 @ 6:00pm
MAX_TUPLES_PER_BAND = 1  #04/16/2025 @ 6:20pm
MAX_TUPLES_PER_BAND = 4  #04/16/2025 @ 7:40pm
MAX_TUPLES_PER_BAND = { #04/16/2025 @ 11:50pm
    1 : 4,
    2 : 16,
    3 : 64,
}

I_k = 5
for D in Ds:
    for seed in seeds:
        np.random.seed(seed)
        synth_dataset_id = f"SYNTH_simple_synthwave_D{D}_seed{seed}"
        synth_dataset_id = f"SYNTH_simple_synthwave_v2_D{D}_seed{seed}"
        synth_dataset_id = f"SYNTH_simple_synthwave_v3_D{D}_seed{seed}"
        synth_dataset_id = f"SYNTH_simple_synthwave_v4_D{D}_seed{seed}"
        synth_dataset_id = f"SYNTH_simple_synthwave_v5_D{D}_seed{seed}"
        synth_dataset_id = f"SYNTH_simple_synthwave_v6_D{D}_seed{seed}"
        
        synth_dataset_id = f"SYNTH_simple_discrete_synthwave_v7_D{D}_Ik{I_k}_seed{seed}"
        synth_dataset_id = f"SYNTH_simple_discrete_synthwave_v8_D{D}_Ik{I_k}_seed{seed}"
        preproc_owner = None #TODO: check if this is fine
        print(synth_dataset_id)
        save_dataset_path = base_save_dataset_path + synth_dataset_id + "/"


        singles = [(i,) for i in range(D)]
        pairs = list(combinations((range(D)),2))
        triples = list(combinations((range(D)),3))

        # singles=singles[:MAX_TUPLES_PER_BAND]
        # pairs=pairs[:MAX_TUPLES_PER_BAND]
        # triples=triples[:MAX_TUPLES_PER_BAND]
        singles=singles[:MAX_TUPLES_PER_BAND[1]]
        pairs=pairs[:MAX_TUPLES_PER_BAND[2]]
        triples=triples[:MAX_TUPLES_PER_BAND[3]]

        all_indices = singles + pairs + triples
        ###all_indices = [(0,), (1,2), (3,4,5),] #just checking
        print(all_indices)



        # X = (np.random.rand(N,D)*2-1)*np.sqrt(3)
        # X = (np.random.rand(N,D)*2-1)*np.sqrt(1)
        X = (np.random.rand(N,D)*2-1)*np.sqrt(4)

        X_int = np.random.randint(I_k,size=(N,D))
        print(X_int.shape)
        X_onehot = np.zeros((N,D*I_k),dtype=int)
        for d in range(D):
            X_onehot[np.arange(N),d*I_k+X_int[:,d]] = 1

        
        beta_dict = {}
        beta_power_bands = {}
        for K in range(3):
            beta_power_bands[K+1] = 0.0
        for tup in all_indices:
            # beta_tt = np.random.randn()
            # beta_dict[tup] = beta_tt
            # beta_power_bands[len(tup)] += beta_tt**2

            
            beta_tt = np.random.randn(*([I_k]*len(tup)))
            beta_dict[tup] = beta_tt
            ############beta_power_bands[len(tup)] += beta_tt**2

            complete_semimasked_U = np.ones_like(beta_tt)
            complete_semimasked_U = complete_semimasked_U / np.sum(complete_semimasked_U)
            # print(complete_semimasked_U)

            purepower = list(powerset(tup))
            # residuals_dict = {}
            # residuals_dict[tup] = torch.zeros_like(theta_tt)


            # for ii in range(len(tup)):
            #     print(np.sum(beta_tt,axis=ii))
                
            currlen=len(tup)
            residual = 0.0
            for purelen in range(currlen-1,-1,-1): 
                for pp,puretup in enumerate(purepower):
                    if len(puretup)==purelen:
                        # print('puretup',puretup)
                        
                        # exttup1 = tuple( [(slice(None) if i in puretup else None) for i in tup] )
                        # exttup2 = tuple( [(slice(None) if i not in puretup else None) for i in tup] )
                        # axes_to_sum = tuple( [ii for ii,i in enumerate(tup) if i in puretup] )

                        # unflattened_U = np.sum(complete_semimasked_U, axis=axes_to_sum)
                        # unflattened_U = unflattened_U[exttup2]
                        # beta_tt_pp = beta_tt * unflattened_U #pure approach
                        # print('beta_tt_pp',beta_tt_pp.shape)
                        exttup1 = tuple( [(slice(None) if i in puretup else None) for i in tup] )
                        exttup2 = tuple( [(slice(None) if i not in puretup else None) for i in tup] )
                        axes_to_sum = tuple( [ii for ii,i in enumerate(tup) if i in puretup] )
                        axes_to_not_sum = tuple( [ii for ii,i in enumerate(tup) if i not in puretup] )

                        unflattened_U = np.sum(complete_semimasked_U, axis=axes_to_sum, keepdims=True)
                        unflattened_beta = np.sum(beta_tt, axis=axes_to_not_sum, keepdims=True)
                        # print('unflattened_U',unflattened_U.shape)
                        # print('unflattened_beta',unflattened_beta.shape)
                        beta_tt_pp = unflattened_beta * unflattened_U

                        sign = 0.0
                        if (purelen-currlen)%2 == 1:
                            sign = 1.0
                        else:
                            sign = -1.0

                        residual += sign * beta_tt_pp
                        # with torch.no_grad():
                        #     self.theta_parameters[str(tup)].grad -= sign*theta_grad_pp
            # beta_tt_adjusted = beta_tt + residual
            beta_tt_adjusted = beta_tt - residual
            beta_power_bands[len(tup)] += np.mean(beta_tt_adjusted**2)

            # print(np.sum(beta_tt_adjusted**2))
            # print(beta_power_bands)

            # for ii in range(len(tup)):
            #     print(np.sum(beta_tt_adjusted,axis=ii))
        
            beta_dict[tup] = beta_tt_adjusted


        
        full_y = 0.0
        # for tup in all_indices:
        # for tup in all_indices[1:]:
        for tup in all_indices[0:]:
            # final_beta_tt = beta_dict[tup] / beta_power_bands[len(tup)] * 2.0 / 3.
            # final_beta_tt = beta_dict[tup] / np.sqrt(beta_power_bands[len(tup)]) * np.sqrt(2.0) / np.sqrt(3.)
            
            # final_beta_tt = beta_dict[tup] / np.sqrt(beta_power_bands[len(tup)]) * np.sqrt(3.0) / np.sqrt(3.) #04/16/2025 @ 7:00pm
            # print('final_beta_tt',final_beta_tt)
            
            # final_beta_tt = beta_dict[tup] / np.sqrt(beta_power_bands[len(tup)]) ###* np.sqrt(3.0) / np.sqrt(2.) #04/16/2025 @ 11:00pm - HUH? needed for discrete ones?
            # 1.0011416020123372
            # nope there were a bunch of bugs causing that
            
            # final_beta_tt = beta_dict[tup] / np.sqrt(beta_power_bands[len(tup)]) / np.sqrt(3.)
            # final_beta_tt = beta_dict[tup] / np.sqrt(beta_power_bands[len(tup)]) / np.sqrt(1.)
            final_beta_tt = beta_dict[tup] / np.sqrt(beta_power_bands[len(tup)]) / np.sqrt(3.)
            
            #REALLY BAD FOR MEMORY
            # preX = np.ones(())
            # for ii,i in enumerate(tup):
            #     print(X[:,i].shape)
            #     print(X[:,i][None].shape)
            #     print(X[:,i][tuple([None])].shape)
            #     preX = preX[...,None] * X[:,i][ tuple([(None)]*(ii)) ]
            #     print('preX',preX.shape)
            # ###preY = np.sin(np.pi*preX)
            # # preY = traingle_sinewave(preX) #starting with v4
            # preY = traingle_sinewave(preX / 2.0) #starting with v5 -- issues with relu spreads?

            preY = np.zeros(X.shape[0])
            for i_tup in product(*([list(range(I_k))]*len(tup))):
                #print(i_tup)

                
                inds = np.ones(X.shape[0],dtype=bool)
                # print(inds[:10])
                for kk,k in enumerate(tup):
                    # print(np.any(X[:,k]==i_tup[kk]))
                    # inds = np.logical_and(inds, X[:,k]==i_tup[kk])
                    inds = np.logical_and(inds, X_onehot[:,I_k*k+i_tup[kk]]==1)
                    # print(inds[:10])
                # print(inds.shape)
                preY[inds] += final_beta_tt[i_tup]
                # print('\tpreY','var',np.var(preY))

            # # plt.scatter(X_onehot[-TST_N:,tup[0]],preY[-TST_N:])
            # plt.scatter(np.matmul(X_onehot[-TST_N:,tup[0]*I_k:tup[0]*I_k+I_k],np.arange(I_k)),preY[-TST_N:])
            # plt.show()

            ###full_y += final_beta_tt * preY
            full_y +=  preY
            print('full_y','var',np.var(full_y))
        print('full_y','var',np.var(full_y))





        readable_labels = {}
        full_readable_labels = {}
        startdim = 0
#         for d in range(D):
        for d in range(-1,D):
            full_info_d = {
                "label" : "X"+str(d+1),
                "startdim" : d*I_k,
                "numdims" : I_k,
                "count" : I_k,
                "encoding" : "disc.onehot",
            }
            if d==-1:
                full_info_d = {
                    "label" : "Y",
                    "startdim" : 0,
                    "numdims" : 1,
                    "encoding" : "cts.raw",
                }
            readable_labels[d] = full_info_d["label"]
            full_readable_labels[d] = full_info_d
        full_readable_labels["task_type"] = "regression"
        full_readable_labels["D0"] = D
        full_readable_labels["D"] = D
        
        
        fake_header_dict = {
            "Preprocessed Datetime" : datetimestr, 
            "dataset_id" : synth_dataset_id,
            "preproc_owner" : preproc_owner,

            #"load_dataset_path" : load_dataset_path,
            #"save_dataset_path" : save_dataset_path,
        }

        trainval_portion, is_test_split_shuffled, shuffle_test_split_seed = None, False, None
        if trainval_portion is None:
            trainval_portion = 0.80
            trainval_portion = TRN_N/N
            assert int(trainval_portion*N)==TRN_N
        fake_header_dict["is_test_split_shuffled"]  = is_test_split_shuffled
        fake_header_dict["shuffle_test_split_seed"] = shuffle_test_split_seed
        fake_header_dict["trainval_portion"]        = trainval_portion
        
        fake_header_dict["datagen_seed"] = seed

        #######label_stuff = (full_readable_labels, datatype_labels, full_readable_labels)
        label_stuff = (readable_labels, full_readable_labels)    
        label_dict = {
            "readable_labels" : label_stuff[0],
            "full_readable_labels" : label_stuff[1],
        }

        if True:
            fullX=X_onehot; fullY=full_y;
            final_save_header(save_dataset_path, fake_header_dict)
            trnvalX,trnvalY,tstX,tstY = final_save_dataset(save_dataset_path, fullX, fullY, 
                                                            trainval_portion, is_test_split_shuffled, shuffle_test_split_seed)

            ''' asdf asdf asdf '''
            ### #### #### #### ###
#             final_save_labels(save_dataset_path, *label_stuff)
            final_save_labels(save_dataset_path, label_dict)

SYNTH_simple_discrete_synthwave_v8_D10_Ik5_seed0
[(0,), (1,), (2,), (3,), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (0, 1, 2), (0, 1, 3), (0, 1, 4), (0, 1, 5), (0, 1, 6), (0, 1, 7), (0, 1, 8), (0, 1, 9), (0, 2, 3), (0, 2, 4), (0, 2, 5), (0, 2, 6), (0, 2, 7), (0, 2, 8), (0, 2, 9), (0, 3, 4), (0, 3, 5), (0, 3, 6), (0, 3, 7), (0, 3, 8), (0, 3, 9), (0, 4, 5), (0, 4, 6), (0, 4, 7), (0, 4, 8), (0, 4, 9), (0, 5, 6), (0, 5, 7), (0, 5, 8), (0, 5, 9), (0, 6, 7), (0, 6, 8), (0, 6, 9), (0, 7, 8), (0, 7, 9), (0, 8, 9), (1, 2, 3), (1, 2, 4), (1, 2, 5), (1, 2, 6), (1, 2, 7), (1, 2, 8), (1, 2, 9), (1, 3, 4), (1, 3, 5), (1, 3, 6), (1, 3, 7), (1, 3, 8), (1, 3, 9), (1, 4, 5), (1, 4, 6), (1, 4, 7), (1, 4, 8), (1, 4, 9), (1, 5, 6), (1, 5, 7), (1, 5, 8), (1, 5, 9), (1, 6, 7), (1, 6, 8), (1, 6, 9), (1, 7, 8), (1, 7, 9), (1, 8, 9)]
(1010000, 10)
full_y var 0.06341104563029273
full_y var 0.16308656526380969
full_y var 0.30213

In [4]:

#         readable_labels = {}
#         full_readable_labels = {}
#         startdim = 0
# #         for d in range(D):
#         for d in range(-1,D):
#             full_info_d = {
#                 "label" : "X"+str(d+1),
#                 "startdim" : d,
#                 "numdims" : 1,
#                 "encoding" : "cts.raw",
#             }
#             if d==-1:
#                 full_info_d = {
#                     "label" : "Y",
#                     "startdim" : d,
#                     "numdims" : 1,
#                     "encoding" : "cts.raw",
#                 }
#             readable_labels[d] = full_info_d["label"]
#             full_readable_labels[d] = full_info_d
#         full_readable_labels["task_type"] = "regression"
#         full_readable_labels["D0"] = D
#         full_readable_labels["D"] = D
        
        
#         fake_header_dict = {
#             "Preprocessed Datetime" : datetimestr, 
#             "dataset_id" : synth_dataset_id,
#             "preproc_owner" : preproc_owner,

#             #"load_dataset_path" : load_dataset_path,
#             #"save_dataset_path" : save_dataset_path,
#         }

#         trainval_portion, is_test_split_shuffled, shuffle_test_split_seed = None, False, None
#         if trainval_portion is None:
#             trainval_portion = 0.80
#             trainval_portion = TRN_N/N
#             assert int(trainval_portion*N)==TRN_N
#         fake_header_dict["is_test_split_shuffled"]  = is_test_split_shuffled
#         fake_header_dict["shuffle_test_split_seed"] = shuffle_test_split_seed
#         fake_header_dict["trainval_portion"]        = trainval_portion
        
#         fake_header_dict["datagen_seed"] = seed

#         #######label_stuff = (full_readable_labels, datatype_labels, full_readable_labels)
#         label_stuff = (readable_labels, full_readable_labels)    
#         label_dict = {
#             "readable_labels" : label_stuff[0],
#             "full_readable_labels" : label_stuff[1],
#         }

#         if True:
#             fullX=X; fullY=full_y;
#             final_save_header(save_dataset_path, fake_header_dict)
#             trnvalX,trnvalY,tstX,tstY = final_save_dataset(save_dataset_path, fullX, fullY, 
#                                                             trainval_portion, is_test_split_shuffled, shuffle_test_split_seed)

#             ''' asdf asdf asdf '''
#             ### #### #### #### ###
# #             final_save_labels(save_dataset_path, *label_stuff)
#             final_save_labels(save_dataset_path, label_dict)